# Assignment 1.2: Benchmarking Code- Aurinder Chohan

## Enviorment Setup

In [10]:
%pip install -q uv

import shutil
import subprocess
from pathlib import Path

ENV = Path("molscribe_env")

if ENV.exists():
    shutil.rmtree(ENV)

subprocess.run(
    ["uv", "venv", "--python", "3.10", str(ENV)],
    check=True
)

PYTHON = str(Path.cwd() / ENV / "bin" / "python")

requirements = """numpy==1.26.4
torch==1.13.1
torchvision==0.14.1
pandas==2.2.3
pymupdf==1.24.14
rdkit==2024.9.6
huggingface-hub==0.27.1
opencv-python==4.8.1.78
setuptools<81
MolScribe==1.1.1
"""

Path("requirements.txt").write_text(requirements)

install = subprocess.run(
    [
        "uv",
        "pip",
        "install",
        "--python",
        PYTHON,
        "-r",
        "requirements.txt"
    ],
    capture_output=True,
    text=True
)

print(install.stdout)
print(install.stderr)

if install.returncode != 0:
    raise RuntimeError(
        "Package installation failed. Review the message above."
    )

test = subprocess.run(
    [
        PYTHON,
        "-c",
        "import sys, torch, pandas, fitz; "
        "from molscribe import MolScribe; "
        "print('Environment setup successful'); "
        "print('Python:', sys.version.split()[0]); "
        "print('PyTorch:', torch.__version__); "
        "print('GPU available:', torch.cuda.is_available()); "
        "print('GPU:', torch.cuda.get_device_name(0) "
        "if torch.cuda.is_available() else 'None')"
    ],
    capture_output=True,
    text=True
)

print(test.stdout)
print(test.stderr)

if test.returncode != 0:
    raise RuntimeError(
        "The packages installed, but the import test failed."
    )


Using Python 3.10.22 environment at: molscribe_env
Resolved 89 packages in 538ms
Prepared 89 packages in 45.02s
Installed 89 packages in 570ms
 + absl-py==2.5.1
 + albumentations==1.1.0
 + anyio==4.15.1
 + beautifulsoup4==4.15.0
 + blinker==1.9.0
 + certifi==2026.7.22
 + charset-normalizer==3.5.2
 + click==8.5.0
 + cloudpickle==3.1.2
 + configargparse==1.8.0
 + exceptiongroup==1.3.1
 + fastcore==2.2.33
 + fastprogress==1.1.6
 + filelock==4.1.0
 + flask==3.1.3
 + fsspec==2026.9.0
 + gensim==4.4.0
 + grpcio==1.84.0
 + h11==0.16.0
 + httpcore2==2.13.1
 + httptools==0.9.0
 + httpx2==2.13.1
 + huggingface-hub==0.27.1
 + idna==3.20
 + imageio==2.38.0
 + itsdangerous==2.2.0
 + jinja2==3.1.6
 + joblib==1.6.0
 + lazy-loader==0.6
 + markdown==3.10.3
 + markupsafe==3.0.4
 + molscribe==1.1.1
 + networkx==3.4.2
 + numpy==1.26.4
 + nvidia-cublas-cu11==11.10.3.66
 + nvidia-cuda-nvrtc-cu11==11.7.99
 + nvidia-cuda-runtime-cu11==11.7.99
 + nvidia-cudnn-cu11==8.5.0.96
 + oauthlib==4.0.0
 + opencv-python

## Data Preparation and Setup

In [12]:
import pandas as pd
from pathlib import Path
from google.colab import files

uploaded = files.upload()

pdf_files = [
    "1.pdf",
    "2.pdf",
    "4.pdf",
    "6.pdf",
    "16.pdf",
    "19.pdf"
]

csv_file = "ground_truth_SMILES_confirmed_1,2,4,6,16,19.csv"

missing_files = [
    filename
    for filename in pdf_files + [csv_file]
    if not Path(filename).exists()
]

if missing_files:
    raise FileNotFoundError(
        f"Missing files: {missing_files}"
    )

ground_truth = pd.read_csv(
    csv_file,
    dtype=str,
    keep_default_na=False,
    encoding="utf-8-sig"
)

ground_truth.columns = ground_truth.columns.str.strip()

required_columns = [
    "paperID",
    "name of molecule",
    "canonical_SMILES"
]

missing_columns = [
    column
    for column in required_columns
    if column not in ground_truth.columns
]

if missing_columns:
    raise ValueError(
        f"Missing CSV columns: {missing_columns}"
    )

ground_truth = ground_truth[required_columns].copy()

for column in required_columns:
    ground_truth[column] = ground_truth[column].str.strip()

ground_truth.to_csv(
    "ground_truth_clean.csv",
    index=False
)

print("Data preparation successful")
print("PDF files found:", len(pdf_files))
print("Ground-truth records:", len(ground_truth))
print("\nRecords per paper:")
print(ground_truth.groupby("paperID").size())

display(ground_truth.head())

Saving 1.pdf to 1.pdf
Saving 2.pdf to 2.pdf
Saving 4.pdf to 4.pdf
Saving 6.pdf to 6.pdf
Saving 16.pdf to 16.pdf
Saving 19.pdf to 19.pdf
Saving ground_truth_SMILES_confirmed_1,2,4,6,16,19.csv to ground_truth_SMILES_confirmed_1,2,4,6,16,19.csv
Data preparation successful
PDF files found: 6
Ground-truth records: 222

Records per paper:
paperID
1      17
16      4
19    105
2      34
4      61
6       1
dtype: int64


,paperID,name of molecule,canonical_SMILES
0,1,AMG925,C[C@H]1CC[C@H](n2c3cnccc3c3cnc(Nc4ccc5c(n4)CCN...
1,1,AST487,CCN1CCN(Cc2ccc(NC(=O)Nc3ccc(Oc4cc(NC)ncn4)cc3)...
2,1,Axitinib,CNC(=O)c1ccccc1Sc1ccc2c(/C=C/c3ccccn3)n[nH]c2c1
3,1,Bosutinib,COc1cc(Nc2c(C#N)cnc3cc(OCCCN4CCN(C)CC4)c(OC)cc...
4,1,Cabozantinib,COc1cc2nccc(Oc3ccc(NC(=O)C4(C(=O)Nc5ccc(F)cc5)...


## Model Setup

In [13]:
import subprocess
from pathlib import Path

PYTHON = str(
    Path.cwd() / "molscribe_env" / "bin" / "python"
)

model_code = """
import torch
from huggingface_hub import hf_hub_download
from molscribe import MolScribe

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

checkpoint = hf_hub_download(
    repo_id="yujieq/MolScribe",
    filename="swin_base_char_aux_1m.pth"
)

model = MolScribe(
    checkpoint,
    device=device
)

print("Model setup successful")
print("Device:", device)
print("Checkpoint:", checkpoint)
"""

Path("model_setup.py").write_text(model_code)

result = subprocess.run(
    [PYTHON, "model_setup.py"],
    capture_output=True,
    text=True
)

print(result.stdout)
print(result.stderr)

if result.returncode != 0:
    raise RuntimeError(
        "Model setup failed. Review the message above."
    )

Model setup successful
Device: cuda
Checkpoint: /root/.cache/huggingface/hub/models--yujieq--MolScribe/snapshots/a0189776b7415b82795c7ee81eed311bf5c8724b/swin_base_char_aux_1m.pth

/content/molscribe_env/lib/python3.10/site-packages/torch/functional.py:504: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at ../aten/src/ATen/native/TensorShape.cpp:3190.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]



## PDF Extraction

In [14]:
import subprocess
from pathlib import Path

PYTHON = str(
    Path.cwd() / "molscribe_env" / "bin" / "python"
)

extraction_code = """
from pathlib import Path
import fitz

pdf_files = [
    "1.pdf",
    "2.pdf",
    "4.pdf",
    "6.pdf",
    "16.pdf",
    "19.pdf"
]

output_folder = Path("extracted_pages")
output_folder.mkdir(exist_ok=True)

for old_image in output_folder.glob("*.png"):
    old_image.unlink()

total_pages = 0

for pdf_file in pdf_files:
    paper_id = Path(pdf_file).stem
    document = fitz.open(pdf_file)

    for page_number, page in enumerate(document, start=1):
        image_path = (
            output_folder
            / f"{paper_id}_page_{page_number}.png"
        )

        page.get_pixmap(
            matrix=fitz.Matrix(2, 2),
            alpha=False
        ).save(image_path)

        total_pages += 1

    document.close()

print("PDF extraction successful")
print("PDF files processed:", len(pdf_files))
print("Page images created:", total_pages)
print("Output folder:", output_folder)
"""

Path("pdf_extraction.py").write_text(
    extraction_code
)

result = subprocess.run(
    [PYTHON, "pdf_extraction.py"],
    capture_output=True,
    text=True
)

print(result.stdout)
print(result.stderr)

if result.returncode != 0:
    raise RuntimeError(
        "PDF extraction failed. Review the message above."
    )

page_images = sorted(
    Path("extracted_pages").glob("*.png")
)

print("Verified PNG files:", len(page_images))
print("First five files:")

for image in page_images[:5]:
    print(image.name)

PDF extraction successful
PDF files processed: 6
Page images created: 104
Output folder: extracted_pages


Verified PNG files: 104
First five files:
16_page_1.png
16_page_10.png
16_page_11.png
16_page_12.png
16_page_13.png


## Inference Pipeline

In [16]:
import subprocess
from pathlib import Path

PYTHON = str(Path.cwd() / "molscribe_env" / "bin" / "python")

inference_code = """
from pathlib import Path
import pandas as pd
import torch
from huggingface_hub import hf_hub_download
from molscribe import MolScribe

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
checkpoint = hf_hub_download("yujieq/MolScribe", "swin_base_char_aux_1m.pth")
model = MolScribe(checkpoint, device=device)
image_files = sorted(Path("extracted_pages").glob("*.png"))
results = []

for number, image_file in enumerate(image_files, start=1):
    try:
        prediction = model.predict_image_file(
            str(image_file),
            return_confidence=True
        )
        confidence = prediction.get("confidence")
        if hasattr(confidence, "item"):
            confidence = confidence.item()
        results.append({
            "image": image_file.name,
            "predicted_smiles": prediction.get("smiles", ""),
            "confidence": confidence,
            "status": "success",
            "error": ""
        })
    except Exception as error:
        results.append({
            "image": image_file.name,
            "predicted_smiles": "",
            "confidence": None,
            "status": "failed",
            "error": str(error)
        })
    print(f"Processed {number} of {len(image_files)}", end="\\r")

predictions = pd.DataFrame(results)
predictions.to_csv("molscribe_predictions.csv", index=False)

print("\\nInference pipeline successful")
print("Images processed:", len(predictions))
print("Successful predictions:", (predictions["status"] == "success").sum())
print("Failed predictions:", (predictions["status"] == "failed").sum())
print("Non-empty SMILES:", predictions["predicted_smiles"].fillna("").str.strip().ne("").sum())
print("Output file: molscribe_predictions.csv")
"""

Path("run_inference.py").write_text(inference_code)

result = subprocess.run(
    [PYTHON, "run_inference.py"],
    capture_output=True,
    text=True
)

print(result.stdout)
print(result.stderr)

if result.returncode != 0:
    raise RuntimeError("Inference failed. Review the message above.")

Processed 1 of 104
Processed 2 of 104
Processed 3 of 104
Processed 4 of 104
Processed 5 of 104
Processed 6 of 104
Processed 7 of 104
Processed 8 of 104
Processed 9 of 104
Processed 10 of 104
Processed 11 of 104
Processed 12 of 104
Processed 13 of 104
Processed 14 of 104
Processed 15 of 104
Processed 16 of 104
Processed 17 of 104
Processed 18 of 104
Processed 19 of 104
Processed 20 of 104
Processed 21 of 104
Processed 22 of 104
Processed 23 of 104
Processed 24 of 104
Processed 25 of 104
Processed 26 of 104
Processed 27 of 104
Processed 28 of 104
Processed 29 of 104
Processed 30 of 104
Processed 31 of 104
Processed 32 of 104
Processed 33 of 104
Processed 34 of 104
Processed 35 of 104
Processed 36 of 104
Processed 37 of 104
Processed 38 of 104
Processed 39 of 104
Processed 40 of 104
Processed 41 of 104
Processed 42 of 104
Processed 43 of 104
Processed 44 of 104
Processed 45 of 104
Processed 46 of 104
Processed 47 of 104
Processed 48 of 104
Processed 49 of 104
Processed 50 of 104
Processed

## Verify Inference Results

In [17]:
import pandas as pd
from pathlib import Path

file = Path("molscribe_predictions.csv")

if not file.exists():
    raise FileNotFoundError("molscribe_predictions.csv was not created.")

predictions = pd.read_csv(file)

required_columns = [
    "image",
    "predicted_smiles",
    "confidence",
    "status",
    "error"
]

missing_columns = [
    column for column in required_columns
    if column not in predictions.columns
]

if missing_columns:
    raise ValueError(f"Missing columns: {missing_columns}")

successful = (predictions["status"] == "success").sum()
failed = (predictions["status"] == "failed").sum()
non_empty = predictions["predicted_smiles"].fillna("").str.strip().ne("").sum()
average_confidence = predictions["confidence"].dropna().mean()

print("Inference verification successful")
print("Prediction rows:", len(predictions))
print("Successful predictions:", successful)
print("Failed predictions:", failed)
print("Non-empty SMILES:", non_empty)
print(f"Average confidence: {average_confidence:.4f}")

display(predictions.head())

Inference verification successful
Prediction rows: 104
Successful predictions: 104
Failed predictions: 0
Non-empty SMILES: 96
Average confidence: 0.4114


,image,predicted_smiles,confidence,status,error
0,16_page_1.png,[2H][2H].[2H][2H],0.409133,success,NaN
1,16_page_10.png,[2H].[2H].[2H].[2H].[2H],0.436243,success,NaN
2,16_page_11.png,[2HH].[2HH].[2HH].[2HH].[2HH].[2HH].[2H],0.548053,success,NaN
3,16_page_12.png,[2H].[2H].[2H].[2H].[2H],0.481252,success,NaN
4,16_page_13.png,[2HH].[2HH].[2HH].[2H],0.492416,success,NaN


## Metrics and Results

In [18]:
import json
import pandas as pd

ground_truth = pd.read_csv("ground_truth_clean.csv")
predictions = pd.read_csv("molscribe_predictions.csv")

non_empty = predictions["predicted_smiles"].fillna("").str.strip().ne("")
successful = predictions["status"].eq("success")
failed = predictions["status"].eq("failed")

summary = {
    "ground_truth_records": len(ground_truth),
    "prediction_records": len(predictions),
    "successful_predictions": int(successful.sum()),
    "failed_predictions": int(failed.sum()),
    "non_empty_predictions": int(non_empty.sum()),
    "non_empty_prediction_rate": float(non_empty.mean()),
    "average_confidence": float(predictions["confidence"].dropna().mean())
}

metrics = pd.DataFrame([summary])
metrics.to_csv("metrics.csv", index=False)

with open("summary.json", "w") as file:
    json.dump(summary, file, indent=4)

print("Metrics and results saved successfully")
display(metrics)

Metrics and results saved successfully


,ground_truth_records,prediction_records,successful_predictions,failed_predictions,non_empty_predictions,non_empty_prediction_rate,average_confidence
0,222,104,104,0,96,0.923077,0.411442


## Eval Metrics

In [23]:
%pip install -q rdkit

import pandas as pd
from rdkit import Chem, DataStructs
from rdkit.Chem import rdFingerprintGenerator

ground_truth = pd.read_csv("ground_truth_clean.csv", dtype=str)
predictions = pd.read_csv("molscribe_predictions.csv")
fingerprint = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2048)

reference = {}

for paper_id, group in ground_truth.groupby("paperID"):
    reference[str(paper_id)] = []
    for smiles in group["canonical_SMILES"]:
        molecule = Chem.MolFromSmiles(str(smiles))
        if molecule is not None:
            canonical = Chem.MolToSmiles(molecule, canonical=True, isomericSmiles=True)
            reference[str(paper_id)].append((molecule, canonical))

results = []

for _, row in predictions.iterrows():
    paper_id = row["image"].split("_page_")[0]
    predicted_smiles = row["predicted_smiles"] if pd.notna(row["predicted_smiles"]) else ""
    predicted_molecule = Chem.MolFromSmiles(str(predicted_smiles)) if predicted_smiles else None
    exact_match = 0
    best_tanimoto = 0.0

    if predicted_molecule is not None and paper_id in reference:
        predicted_canonical = Chem.MolToSmiles(
            predicted_molecule,
            canonical=True,
            isomericSmiles=True
        )
        predicted_fp = fingerprint.GetFingerprint(predicted_molecule)

        for true_molecule, true_canonical in reference[paper_id]:
            if predicted_canonical == true_canonical:
                exact_match = 1
            true_fp = fingerprint.GetFingerprint(true_molecule)
            similarity = DataStructs.TanimotoSimilarity(predicted_fp, true_fp)
            best_tanimoto = max(best_tanimoto, similarity)

    results.append({
        "image": row["image"],
        "paperID": paper_id,
        "predicted_smiles": predicted_smiles,
        "valid_prediction": predicted_molecule is not None,
        "exact_match_any_reference": exact_match,
        "best_tanimoto_same_paper": best_tanimoto
    })

evaluation = pd.DataFrame(results)
evaluation.to_csv("evaluation_results.csv", index=False)

evaluation_metrics = pd.DataFrame([{
    "pages_evaluated": len(evaluation),
    "valid_predictions": int(evaluation["valid_prediction"].sum()),
    "exact_matches": int(evaluation["exact_match_any_reference"].sum()),
    "exact_match_rate": float(evaluation["exact_match_any_reference"].mean()),
    "average_best_tanimoto": float(evaluation["best_tanimoto_same_paper"].mean())
}])

evaluation_metrics.to_csv("evaluation_metrics.csv", index=False)

print("Evaluation metrics saved successfully")
display(evaluation_metrics)

Evaluation metrics saved successfully


,pages_evaluated,valid_predictions,exact_matches,exact_match_rate,average_best_tanimoto
0,104,81,0,0.0,0.006333


## Results Summary

In [24]:
import json
import pandas as pd

metrics = pd.read_csv("metrics.csv")
evaluation = pd.read_csv("evaluation_metrics.csv")

summary = {
    "ground_truth_records": int(metrics.loc[0, "ground_truth_records"]),
    "prediction_records": int(metrics.loc[0, "prediction_records"]),
    "successful_predictions": int(metrics.loc[0, "successful_predictions"]),
    "failed_predictions": int(metrics.loc[0, "failed_predictions"]),
    "non_empty_predictions": int(metrics.loc[0, "non_empty_predictions"]),
    "average_confidence": float(metrics.loc[0, "average_confidence"]),
    "pages_evaluated": int(evaluation.loc[0, "pages_evaluated"]),
    "valid_predictions": int(evaluation.loc[0, "valid_predictions"]),
    "exact_matches": int(evaluation.loc[0, "exact_matches"]),
    "exact_match_rate": float(evaluation.loc[0, "exact_match_rate"]),
    "average_best_tanimoto": float(evaluation.loc[0, "average_best_tanimoto"])
}

with open("final_summary.json", "w") as file:
    json.dump(summary, file, indent=4)

print("Final Benchmark Summary")
print("=" * 42)
print("Ground-truth records:", summary["ground_truth_records"])
print("Prediction records:", summary["prediction_records"])
print("Successful predictions:", summary["successful_predictions"])
print("Failed predictions:", summary["failed_predictions"])
print("Non-empty predictions:", summary["non_empty_predictions"])
print(f"Average confidence: {summary['average_confidence']:.4f}")
print("Pages evaluated:", summary["pages_evaluated"])
print("Valid SMILES predictions:", summary["valid_predictions"])
print("Exact matches:", summary["exact_matches"])
print(f"Exact-match rate: {summary['exact_match_rate']:.4f}")
print(f"Average best Tanimoto similarity: {summary['average_best_tanimoto']:.6f}")
print("\nSaved: final_summary.json")

Final Benchmark Summary
Ground-truth records: 222
Prediction records: 104
Successful predictions: 104
Failed predictions: 0
Non-empty predictions: 96
Average confidence: 0.4114
Pages evaluated: 104
Valid SMILES predictions: 81
Exact matches: 0
Exact-match rate: 0.0000
Average best Tanimoto similarity: 0.006333

Saved: final_summary.json


## Download Submission Files

In [25]:
from google.colab import files

output_files = [
    "requirements.txt",
    "ground_truth_clean.csv",
    "molscribe_predictions.csv",
    "metrics.csv",
    "summary.json",
    "evaluation_results.csv",
    "evaluation_metrics.csv",
    "final_summary.json"
]

for filename in output_files:
    files.download(filename)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>